In [ ]:
!pip install requirements.txt
!git clone https://github.com/doszilab/AIUPred


In [ ]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
sys.path.append('/home/ilnitsky/tools/AIUPred')
import aiupred_lib

# PARAMETERS
threshold_positive = 1.0  # Threshold for positive IDR detection
threshold_negative = 1.0  # Threshold for negative IDR detection
min_length = 7          # Minimum length for IDR regions
pH = 7.0                # pH for charge calculation
max_gap = 20            # Maximum gap to connect similar charge IDRs
charge_window_size = 6  # Sliding window size for charge/NCPR calculation
max_region_length = 80  # Maximum length of regions to analyze

names = []
sequences = []
file_path = "/home/ilnitsky/NPM/45_Species/fasta_sequences/ew.fasta"
with open(file_path, "r") as f: 
    lines = f.readlines()
    for line in lines:
        if line.startswith(">"):
            names.append(line.strip().lstrip(">"))
        else:
            sequences.append(line.strip())

sequences4plot = sequences
names4plot = names

# cumulative charge array
# charge is calculated based on fraction of deprotonated and protonated residues
def calculate_cumulative_charge(sequence, pH=pH):
    pKa_values = {
        'D': 3.9,  'E': 4.3,   'H': 6.0,   'C': 8.3,   'Y': 10.1,   'K': 10.5,    'R': 12.5  
    }
    cumulative_charge = np.zeros(len(sequence) + 1)
    for i, aa in enumerate(sequence, 1):
        charge = 0.0
        if aa in pKa_values:
            if aa in ['D', 'E', 'C', 'Y']:
                fraction_deprotonated = 1 / (1 + 10**(pKa_values[aa] - pH))
                charge -= fraction_deprotonated
            elif aa in ['K', 'H', 'R']:
                fraction_protonated = 1 / (1 + 10**(pH - pKa_values[aa]))
                charge += fraction_protonated
        cumulative_charge[i] = cumulative_charge[i-1] + charge
    return cumulative_charge

# FUNCTION TO GET IDR REGIONS AND METRIC PROFILES
def get_idr_regions_and_profiles(sequence, threshold_positive=threshold_positive, threshold_negative=threshold_negative, min_length=min_length, pH=pH, max_gap=max_gap, charge_window_size=charge_window_size, max_region_length=max_region_length):
    max_len = len(sequence)
    idr_regions = {"sqrt": {"positive": [], "negative": []},
                   "squared_sqrt": {"positive": [], "negative": []},
                   "cbrt": {"positive": [], "negative": []}}
    profiles = {
        'sqrt': np.zeros(max_len),
        'squared_sqrt': np.zeros(max_len),
        'cbrt': np.zeros(max_len)
    }
    
    # Precompute cumulative charge
    cumulative_charge = calculate_cumulative_charge(sequence)
    
    # Calculate NCPR with sliding window
    ncpr_x = []
    ncpr_y = []
    for i in range(max_len - charge_window_size + 1):
        window_start = i + 1
        window_end = i + charge_window_size
        charge = cumulative_charge[window_end] - cumulative_charge[window_start - 1]
        ncpr = charge / charge_window_size
        ncpr_x.append(window_start + (charge_window_size - 1) / 2)  # Center of window
        ncpr_y.append(ncpr)
    
    # Calculate metrics for regions up to max_region_length
    for start in range(1, max_len + 1):
        for length in range(min_length, min(max_region_length + 1, max_len - start + 2)):
            end = start + length - 1
            charge = cumulative_charge[end] - cumulative_charge[start - 1]
            region_seq = sequence[start-1:end]
            
            # Compute all metrics
            metrics = {
                'sqrt': abs(charge) / np.sqrt(length),
                'squared_sqrt': (charge ** 2) / np.sqrt(length),
                'cbrt': abs(charge) / (length ** (1/3))
            }
            for metric_name, metric_value in metrics.items():
                if charge > 0 and metric_value > profiles[metric_name][start-1]:
                    profiles[metric_name][start-1] = metric_value
                elif charge < 0 and metric_value > profiles[metric_name][start-1]:
                    profiles[metric_name][start-1] = metric_value
                
                # IDR detection based on each metric's profile value with separate thresholds
                if charge > 0 and metric_value > threshold_positive:
                    idr_regions[metric_name]["positive"].append((start, end, region_seq, metric_value))
                elif charge < 0 and metric_value > threshold_negative:
                    idr_regions[metric_name]["negative"].append((start, end, region_seq, metric_value))
    
    # Debug: Check if IDRs were detected for any metric
    for metric_name in idr_regions:
        if not idr_regions[metric_name]["positive"] and not idr_regions[metric_name]["negative"]:
            print(f"Warning: No IDRs detected for {sequence[:20]}... (len={len(sequence)}) using {metric_name} metric. ")
    
    # Filter out overlapping and handle nested splitting for each metric
    for metric_name in idr_regions:
        def filter_and_split_regions(pos_regions, neg_regions):
            all_regions = pos_regions + neg_regions
            non_overlapping = []
            
            while all_regions:
                r1 = max(all_regions, key=lambda x: x[3])  # Select region with highest metric
                all_regions.remove(r1)
                start1, end1, seq1, metric1 = r1
                overlap = False
                
                for r2 in non_overlapping:
                    start2, end2, _, _ = r2
                    if not (end1 < start2 or start1 > end2):  # Check for overlap
                        overlap = True
                        # If r1 is larger and contains r2 of opposite charge, split r1
                        if (start1 <= start2 and end1 >= end2) and (
                            (r1 in pos_regions and r2 in neg_regions) or (r1 in neg_regions and r2 in pos_regions)
                        ):
                            if start1 < start2:
                                new_seq = sequence[start1-1:start2-1]
                                new_charge = cumulative_charge[start2-1] - cumulative_charge[start1-1]
                                new_length = start2 - start1
                                new_metric = (new_charge ** 2) / np.sqrt(new_length) if metric_name == 'squared_sqrt' else abs(new_charge) / (np.sqrt(new_length) if metric_name == 'sqrt' else new_length ** (1/3) if metric_name == 'cbrt' else new_length)
                                if new_metric > (threshold_positive if new_charge > 0 else threshold_negative) and len(new_seq) >= min_length:
                                    all_regions.append((start1, start2-1, new_seq, new_metric))
                            if end1 > end2:
                                new_seq = sequence[end2-1:end1]
                                new_charge = cumulative_charge[end1] - cumulative_charge[end2-1]
                                new_length = end1 - end2 + 1
                                new_metric = (new_charge ** 2) / np.sqrt(new_length) if metric_name == 'squared_sqrt' else abs(new_charge) / (np.sqrt(new_length) if metric_name == 'sqrt' else new_length ** (1/3) if metric_name == 'cbrt' else new_length)
                                if new_metric > (threshold_positive if new_charge > 0 else threshold_negative) and len(new_seq) >= min_length:
                                    all_regions.append((end2, end1, new_seq, new_metric))
                        break
                
                if not overlap:
                    non_overlapping.append(r1)
            
            return {'positive': [r for r in non_overlapping if r in pos_regions], 
                    'negative': [r for r in non_overlapping if r in neg_regions]}

        idr_regions[metric_name] = filter_and_split_regions(idr_regions[metric_name]["positive"], idr_regions[metric_name]["negative"])
    
    # Connect similar charge IDRs if gap is less than max_gap
    for metric_name in idr_regions:
        pos_regions = idr_regions[metric_name]["positive"]
        neg_regions = idr_regions[metric_name]["negative"]
        
        # Sort regions by start position
        pos_regions.sort(key=lambda x: x[0])
        neg_regions.sort(key=lambda x: x[0])
        
        # Connect positive regions
        i = 0
        while i < len(pos_regions) - 1:
            curr_end = pos_regions[i][1]
            next_start = pos_regions[i + 1][0]
            if next_start - curr_end <= max_gap:
                new_start = pos_regions[i][0]
                new_end = pos_regions[i + 1][1]
                new_seq = sequence[new_start-1:new_end]
                new_metric = max(pos_regions[i][3], pos_regions[i + 1][3])  # Use max metric value
                pos_regions[i] = (new_start, new_end, new_seq, new_metric)
                pos_regions.pop(i + 1)
            else:
                i += 1
        
        # Connect negative regions
        i = 0
        while i < len(neg_regions) - 1:
            curr_end = neg_regions[i][1]
            next_start = neg_regions[i + 1][0]
            if next_start - curr_end <= max_gap:
                new_start = neg_regions[i][0]
                new_end = neg_regions[i + 1][1]
                new_seq = sequence[new_start-1:new_end]
                new_metric = max(neg_regions[i][3], neg_regions[i + 1][3])  # Use max metric value
                neg_regions[i] = (new_start, new_end, new_seq, new_metric)
                neg_regions.pop(i + 1)
            else:
                i += 1
        
        idr_regions[metric_name]["positive"] = pos_regions
        idr_regions[metric_name]["negative"] = neg_regions
    
    return idr_regions, profiles, ncpr_x, ncpr_y

# Function to calculate net charge for a region using pKa and pH (used for validation, can be removed if cumulative_charge is sufficient)
def calculate_net_charge(sequence, start, end, pH=pH):
    region = sequence[start-1:end]  # 0-based indexing adjustment
    net_charge = 0.0
    
    pKa_values = {
        'D': 3.9,  # Aspartic acid
        'E': 4.3,  # Glutamic acid
        'H': 6.0,  # Histidine
        'C': 8.3,  # Cysteine
        'Y': 10.1, # Tyrosine
        'K': 10.5, # Lysine
        'R': 12.5  # Arginine
    }
    
    for aa in region:
        if aa in pKa_values:
            if aa in ['D', 'E', 'H', 'C', 'Y']:
                fraction_deprotonated = 1 / (1 + 10**(pKa_values[aa] - pH))
                net_charge -= fraction_deprotonated
            elif aa in ['K', 'R']:
                fraction_protonated = 1 / (1 + 10**(pH - pKa_values[aa]))
                net_charge += fraction_protonated
    
    return net_charge

fig, axs = plt.subplots(len(sequences4plot), 3, figsize=(36, 4*len(sequences4plot)), squeeze=False)  # Changed to 3 columns
axs = axs.ravel()

# PRINT SECTIONS
print("="*80)
print(f"IDR REGIONS (Positive Threshold={threshold_positive}, Negative Threshold={threshold_negative} for each metric, Min Length={min_length}aa, pH={pH}, Max Gap={max_gap}, Charge Window Size={charge_window_size}, Max Region Length={max_region_length})")
print("="*80)

for i, (sequence, name) in enumerate(zip(sequences4plot, names4plot)):
    print(f"\n{name}")
    print("-"*len(name))
    
    disorder_prediction = aiupred_lib.aiupred_disorder(sequence)
    
    # Get NCPR data and IDR regions with profiles
    try:
        idr_regions, profiles, ncpr_x, ncpr_y = get_idr_regions_and_profiles(sequence)
    except:
        print(f"Error with sequence {i}: {name}")
        continue
    
    # Print IDR regions for each metric
    for metric_name in idr_regions:
        print(f"\n{metric_name.upper()} Metric IDRs:")
        if idr_regions[metric_name]["positive"]:
            for start, end, seq_region, metric in idr_regions[metric_name]["positive"]:
                print(f"+IDR {start}-{end} (Metric: {metric:.2f}) {seq_region}")
        if idr_regions[metric_name]["negative"]:
            for start, end, seq_region, metric in idr_regions[metric_name]["negative"]:
                print(f"-IDR {start}-{end} (Metric: {metric:.2f}) {seq_region}")

print("\n" + "="*80)

# PLOTTING
for i, (sequence, name) in enumerate(zip(sequences4plot, names4plot)):
    print(i, name)
    disorder_prediction = aiupred_lib.aiupred_disorder(sequence)
    positions = range(1, len(sequence) + 1)
    
    # Get NCPR data and IDR regions with profiles
    try:
        idr_regions, profiles, ncpr_x, ncpr_y = get_idr_regions_and_profiles(sequence)
    except:
        print(f"Error with sequence {i}: {name}")
        continue

    # Plot for each metric profile
    for j, metric_name in enumerate(['sqrt', 'squared_sqrt', 'cbrt']):
        ax = axs[i * 3 + j]  # Adjusted indexing for 3 columns

        # PLOT 2: VERTICAL LINES for disorder (individual disorder >0.6 = blue)
        for pos, disorder in zip(positions, disorder_prediction):
            color = 'blue' if disorder > 0.6 else 'gray'
            alpha = 0.3
            ax.axvline(x=pos, color=color, alpha=alpha, linewidth=0.5)

        # PLOT 4: IDR BORDERS specific to each metric
        for start, end, _, _ in idr_regions[metric_name]["positive"]:
            ax.axvline(x=start, color='green', linewidth=2, alpha=0.8, linestyle='--')
            ax.axvline(x=end, color='green', linewidth=2, alpha=0.8, linestyle='--')
            ax.axvspan(start, end, color='green', alpha=0.1)
        for start, end, _, _ in idr_regions[metric_name]["negative"]:
            ax.axvline(x=start, color='purple', linewidth=2, alpha=0.8, linestyle='--')
            ax.axvline(x=end, color='purple', linewidth=2, alpha=0.8, linestyle='--')
            ax.axvspan(start, end, color='purple', alpha=0.1)

        # PLOT 5: NCPR (bars) for reference
        ax2 = ax.twinx()
        for x, y in zip(ncpr_x, ncpr_y):
            color = 'green' if y >= 0 else 'red'
            ax2.bar(x, y, width=1, color=color, alpha=0.5, align='center')

        # Set axis properties
        ax.set_ylim([0, 1])  # Adjusted to focus on disorder and IDR visualization
        ax2.set_ylim([-1.0, 1.0])
        ax.set_xticks([])
        ax2.set_xticks([])
        ax.set_yticks([0, 0.5, 1])
        ax2.set_yticks([])
        
        ax.set_title(f'{name} - {metric_name.capitalize()} Profile')
        ax.set_ylabel('Disorder Score', color='black')
        if j == 0:
            ax.legend(['Disorder'], loc='upper right')

plt.tight_layout()
plt.gcf().set_size_inches(36, 4*len(sequences4plot))  # Adjusted for 3 plots per row

plt.show()

IDR REGIONS (Positive Threshold=1.0, Negative Threshold=1.0 for each metric, Min Length=7aa, pH=7.0, Max Gap=20, Charge Window Size=6, Max Region Length=80)

Monosiga brevicollis|MONBRDRAFT_39241|A9VD53|IDR|123-320
--------------------------------------------------------

SQRT Metric IDRs:
+IDR 82-95 (Metric: 1.60) KRGKGASNSNKRAK
+IDR 180-189 (Metric: 1.58) KKAAKKTAAK
-IDR 19-76 (Metric: 3.92) DSDDEDESDEDDLDDDDHLDSQTGRVVTLEDSSDQEDDVDSPIIEVLEDDDDLEEAPE
-IDR 97-168 (Metric: 5.77) DEEDDEEDEEDDVAGQEESKLQLTMDVMEDEDDEDDEDDEDAELDLSMTVPDDEDDEDDEDDEELDDDEEED

SQUARED_SQRT Metric IDRs:
+IDR 82-95 (Metric: 9.62) KRGKGASNSNKRAK
+IDR 175-189 (Metric: 9.29) KTPVTKKAAKKTAAK
-IDR 2-76 (Metric: 124.04) EQVQETMMQLQEHGILGDSDDEDESDEDDLDDDDHLDSQTGRVVTLEDSSDQEDDVDSPIIEVLEDDDDLEEAPE
-IDR 97-168 (Metric: 282.20) DEEDDEEDEEDDVAGQEESKLQLTMDVMEDEDDEDDEDDEDAELDLSMTVPDDEDDEDDEDDEELDDDEEED

CBRT Metric IDRs:
+IDR 82-95 (Metric: 2.49) KRGKGASNSNKRAK
+IDR 175-189 (Metric: 2.43) KTPVTKKAAKKTAAK
-IDR 2-76 (Metric: 7.77